# Building Footprint Segmentation (Massachusetts Buildings)

Binary semantic segmentation of aerial imagery using PyTorch, segmentation_models_pytorch, and Albumentations to predict building masks.

## Project Overview

This notebook builds an end-to-end footprint segmenter on the Massachusetts Buildings dataset. It downloads the dataset with kagglehub, validates the image and mask pairs, trains a lightweight U-Net baseline with a pretrained encoder, and reports IoU, Dice, and boundary-sensitive metrics.

## Learning Objectives

1. Build a paired image-mask segmentation dataset for aerial imagery.
2. Train an SMP U-Net model with Albumentations preprocessing.
3. Evaluate binary segmentation with IoU, Dice, and boundary F1.
4. Diagnose where predictions fail along building edges, not just in region overlap.

## Problem Statement

Given an aerial RGB image, predict a binary mask where building pixels are 1 and background pixels are 0.

## Why This Project Matters

Accurate building footprints matter in mapping, urban planning, disaster response, taxation, and change detection. Boundary quality matters because small geometric errors can create large downstream problems in area estimates and polygon extraction.

## Dataset Overview

Dataset: Massachusetts Buildings Dataset

- 151 aerial tiles at 1500 x 1500 pixels
- Official split in the Kaggle mirror: 137 train, 4 validation, 10 test
- Paired RGB images and binary building masks
- Binary labels encoded as black and white PNG masks

## Dataset Source and License Notes

Kaggle mirror: https://www.kaggle.com/datasets/balraj98/massachusetts-buildings-dataset

This notebook uses the Kaggle mirror via kagglehub. Review the dataset page for usage terms before reuse outside learning or research workflows.

## Environment Setup

Install the core packages in the active environment before running the notebook end to end.

```bash
pip install torch torchvision segmentation-models-pytorch albumentations kagglehub pillow opencv-python matplotlib tqdm
```

The notebook uses CUDA automatically when available and falls back to CPU otherwise.

In [ ]:
import json
import os
import random
from pathlib import Path

import albumentations as A
import cv2
import kagglehub
import matplotlib
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from matplotlib import pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

matplotlib.use("Agg")

print("torch:", torch.__version__)
print("smp:", smp.__version__)
print("albumentations:", A.__version__)
print("cuda:", torch.cuda.is_available())

## Configuration / Constants

These settings keep the notebook reproducible and small enough to validate locally. The model uses a lighter encoder than the previous road-scene notebook because the Massachusetts tiles are much larger and the repo validation run must finish on CPU if needed.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = Path.cwd()
ARTIFACTS_DIR = BASE_DIR / "artifacts"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
DATA_DIR = BASE_DIR / "data"
for folder in [ARTIFACTS_DIR, CHECKPOINT_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_ID = "balraj98/massachusetts-buildings-dataset"
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 4
LEARNING_RATE = 1e-3
ENCODER_NAME = "mobilenet_v2"
ENCODER_WEIGHTS = "imagenet"
BOUNDARY_KERNEL = 3
BOUNDARY_TOLERANCE = 2
NUM_WORKERS = 0

print({
    "device": DEVICE,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "encoder": ENCODER_NAME,
})

## Dataset Download and Loading

The notebook downloads the Massachusetts Buildings dataset from Kaggle with `kagglehub`. The real image and label files live under the extracted `png/` directory in official `train`, `val`, and `test` splits.

In [ ]:
dataset_root = Path(kagglehub.dataset_download(DATASET_ID))
png_root = dataset_root / "png"

split_dirs = {
    "train": (png_root / "train", png_root / "train_labels"),
    "val": (png_root / "val", png_root / "val_labels"),
    "test": (png_root / "test", png_root / "test_labels"),
}

for split, (image_dir, label_dir) in split_dirs.items():
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f"Missing expected split folders for {split}: {image_dir} / {label_dir}")


def list_pairs(image_dir: Path, label_dir: Path):
    image_paths = sorted(image_dir.glob("*.png"))
    label_paths = sorted(label_dir.glob("*.png"))
    if len(image_paths) != len(label_paths):
        raise ValueError(f"Mismatched counts in {image_dir} and {label_dir}")
    pairs = []
    for image_path in image_paths:
        label_path = label_dir / image_path.name
        if not label_path.exists():
            raise FileNotFoundError(f"Missing label for {image_path.name}")
        pairs.append((image_path, label_path))
    return pairs

split_pairs = {split: list_pairs(image_dir, label_dir) for split, (image_dir, label_dir) in split_dirs.items()}
split_counts = {split: len(pairs) for split, pairs in split_pairs.items()}

print("Dataset root:", dataset_root)
print(json.dumps(split_counts, indent=2))
(DATA_DIR / "dataset_info.json").write_text(
    json.dumps({"dataset_id": DATASET_ID, "dataset_root": str(dataset_root), "split_counts": split_counts}, indent=2),
    encoding="utf-8",
)

## Data Validation Checks

Before training, verify that every split has matching image-mask pairs and that the masks are truly binary.

## Exploratory Data Analysis

We inspect a few training examples and summarize how much of each tile is covered by buildings.

## Target Analysis

This is a binary segmentation task. The target distribution is the building coverage ratio per image, which is useful because empty or low-coverage masks can make IoU and Dice unstable on small validation sets.

In [ ]:
def load_rgb_image(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def load_binary_mask(path: Path) -> np.ndarray:
    mask = np.array(Image.open(path).convert("L"))
    return (mask > 127).astype(np.uint8)


def overlay_mask(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    overlay = image.copy()
    overlay[mask == 1] = (0.55 * overlay[mask == 1] + 0.45 * np.array([255, 80, 80])).astype(np.uint8)
    return overlay


train_coverages = []
mask_unique_values = set()
for _, mask_path in split_pairs["train"]:
    mask = load_binary_mask(mask_path)
    train_coverages.append(float(mask.mean()))
    mask_unique_values.update(np.unique(mask).tolist())

if mask_unique_values != {0, 1}:
    raise ValueError(f"Unexpected mask values: {mask_unique_values}")

print("Average train building coverage:", round(float(np.mean(train_coverages)), 4))
print("Min/Max train coverage:", round(float(np.min(train_coverages)), 4), round(float(np.max(train_coverages)), 4))
print("Mask values:", sorted(mask_unique_values))

sample_pairs = split_pairs["train"][:4]
fig, axes = plt.subplots(len(sample_pairs), 3, figsize=(12, 3 * len(sample_pairs)))
for row, (image_path, mask_path) in enumerate(sample_pairs):
    image = load_rgb_image(image_path)
    mask = load_binary_mask(mask_path)
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"RGB\n{image_path.name}")
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title(f"Mask\ncoverage={mask.mean():.3f}")
    axes[row, 2].imshow(overlay_mask(image, mask))
    axes[row, 2].set_title("Overlay")
    for col in range(3):
        axes[row, col].axis("off")
fig.tight_layout()
eda_grid_path = ARTIFACTS_DIR / "eda_samples.png"
fig.savefig(eda_grid_path, dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_coverages, bins=12, color="#2c7fb8", edgecolor="black")
ax.set_title("Train Split Building Coverage Ratio")
ax.set_xlabel("Building pixels / total pixels")
ax.set_ylabel("Image count")
fig.tight_layout()
coverage_hist_path = ARTIFACTS_DIR / "target_coverage_histogram.png"
fig.savefig(coverage_hist_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print("Saved:", eda_grid_path.name)
print("Saved:", coverage_hist_path.name)

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def build_transforms(train: bool) -> A.Compose:
    ops = [
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
    ]
    if train:
        ops.extend(
            [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
            ]
        )
    ops.extend([A.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()), ToTensorV2()])
    return A.Compose(ops)


class MassachusettsBuildingsDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image = load_rgb_image(image_path)
        mask = load_binary_mask(mask_path)
        transformed = self.transform(image=image, mask=mask)
        image_tensor = transformed["image"]
        mask_tensor = transformed["mask"].float().unsqueeze(0)
        return image_tensor, mask_tensor, image_path.name


train_ds = MassachusettsBuildingsDataset(split_pairs["train"], build_transforms(train=True))
val_ds = MassachusettsBuildingsDataset(split_pairs["val"], build_transforms(train=False))
test_ds = MassachusettsBuildingsDataset(split_pairs["test"], build_transforms(train=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))
print("Feature engineering: none beyond resize, normalization, and geometric augmentation.")

## Train / Validation / Test Split Strategy

We keep the official dataset split instead of resampling. That preserves the intended benchmark and avoids leakage across nearby aerial tiles.

## Preprocessing and Augmentation Strategy

- Resize every image and mask to a shared training size.
- Use nearest-neighbor interpolation for masks so building boundaries are not blurred.
- Apply light geometric augmentation on the training split only.
- Normalize with ImageNet statistics because the encoder starts from ImageNet weights.

## Baseline Approach

Before training a neural segmenter, we compute a weak classical baseline using grayscale Otsu thresholding plus morphology. It is not expected to be competitive, but it gives a sanity-check floor.

## Main Model / Workflow

The main model is `smp.Unet` with a pretrained `mobilenet_v2` encoder. This is a pragmatic tradeoff: much lighter than a ResNet-34 encoder, but still strong enough to learn building shapes from the 137 training tiles.

In [ ]:
def grayscale_baseline_mask(image_rgb: np.ndarray, reference_coverage: float) -> np.ndarray:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, raw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    candidate_a = (raw > 0).astype(np.uint8)
    candidate_b = 1 - candidate_a

    def clean(mask: np.ndarray) -> np.ndarray:
        kernel = np.ones((3, 3), np.uint8)
        opened = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, kernel)
        closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
        return closed.astype(np.uint8)

    candidate_a = clean(candidate_a)
    candidate_b = clean(candidate_b)
    if abs(candidate_a.mean() - reference_coverage) <= abs(candidate_b.mean() - reference_coverage):
        return candidate_a
    return candidate_b


def binary_iou_dice(pred_mask: np.ndarray, true_mask: np.ndarray):
    pred_bool = pred_mask.astype(bool)
    true_bool = true_mask.astype(bool)
    intersection = np.logical_and(pred_bool, true_bool).sum()
    union = np.logical_or(pred_bool, true_bool).sum()
    pred_sum = pred_bool.sum()
    true_sum = true_bool.sum()
    iou = float(intersection / union) if union > 0 else 1.0
    dice = float((2 * intersection) / (pred_sum + true_sum)) if (pred_sum + true_sum) > 0 else 1.0
    return iou, dice


def boundary_mask(mask: np.ndarray, kernel_size: int = BOUNDARY_KERNEL) -> np.ndarray:
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    gradient = cv2.morphologyEx((mask.astype(np.uint8) * 255), cv2.MORPH_GRADIENT, kernel)
    return (gradient > 0).astype(np.uint8)


def boundary_f1_score(pred_mask: np.ndarray, true_mask: np.ndarray, tolerance: int = BOUNDARY_TOLERANCE) -> float:
    pred_boundary = boundary_mask(pred_mask)
    true_boundary = boundary_mask(true_mask)
    if pred_boundary.sum() == 0 and true_boundary.sum() == 0:
        return 1.0
    if pred_boundary.sum() == 0 or true_boundary.sum() == 0:
        return 0.0
    kernel = np.ones((2 * tolerance + 1, 2 * tolerance + 1), np.uint8)
    pred_dilated = cv2.dilate(pred_boundary, kernel, iterations=1)
    true_dilated = cv2.dilate(true_boundary, kernel, iterations=1)
    precision = (pred_boundary & true_dilated).sum() / max(pred_boundary.sum(), 1)
    recall = (true_boundary & pred_dilated).sum() / max(true_boundary.sum(), 1)
    if precision + recall == 0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))


def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    image = image_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    image = (image * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    return (image * 255).astype(np.uint8)


def segmentation_loss(logits: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
    bce = nn.functional.binary_cross_entropy_with_logits(logits, masks)
    dice = smp.losses.DiceLoss(mode="binary")(logits, masks)
    return bce + dice


def evaluate_model(model: nn.Module, loader: DataLoader, criterion=None):
    model.eval()
    losses = []
    records = []
    with torch.no_grad():
        for images, masks, names in loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)
            logits = model(images)
            if criterion is not None:
                losses.append(float(criterion(logits, masks).item()))
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            for index, name in enumerate(names):
                pred_mask = preds[index, 0].detach().cpu().numpy().astype(np.uint8)
                true_mask = masks[index, 0].detach().cpu().numpy().astype(np.uint8)
                iou, dice = binary_iou_dice(pred_mask, true_mask)
                boundary_f1 = boundary_f1_score(pred_mask, true_mask)
                records.append(
                    {
                        "name": name,
                        "iou": iou,
                        "dice": dice,
                        "boundary_f1": boundary_f1,
                        "coverage_error": float(abs(pred_mask.mean() - true_mask.mean())),
                    }
                )
    summary = {
        "loss": float(np.mean(losses)) if losses else None,
        "iou": float(np.mean([record["iou"] for record in records])) if records else 0.0,
        "dice": float(np.mean([record["dice"] for record in records])) if records else 0.0,
        "boundary_f1": float(np.mean([record["boundary_f1"] for record in records])) if records else 0.0,
        "coverage_error": float(np.mean([record["coverage_error"] for record in records])) if records else 0.0,
        "records": records,
    }
    return summary


reference_coverage = float(np.mean(train_coverages))
baseline_records = []
for image_path, mask_path in split_pairs["val"]:
    image = load_rgb_image(image_path)
    true_mask = load_binary_mask(mask_path)
    pred_mask = grayscale_baseline_mask(image, reference_coverage)
    iou, dice = binary_iou_dice(pred_mask, true_mask)
    baseline_records.append(
        {
            "name": image_path.name,
            "iou": iou,
            "dice": dice,
            "boundary_f1": boundary_f1_score(pred_mask, true_mask),
        }
    )

baseline_summary = {
    "iou": float(np.mean([record["iou"] for record in baseline_records])),
    "dice": float(np.mean([record["dice"] for record in baseline_records])),
    "boundary_f1": float(np.mean([record["boundary_f1"] for record in baseline_records])),
}
print("Validation baseline:", json.dumps(baseline_summary, indent=2))

model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=1,
).to(DEVICE)

print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_checkpoint_path = CHECKPOINT_DIR / "best_building_unet.pth"
best_val_iou = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for images, masks, _ in progress:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = segmentation_loss(logits, masks)
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss.item()))
        progress.set_postfix(train_loss=f"{loss.item():.4f}")

    scheduler.step()
    val_summary = evaluate_model(model, val_loader, criterion=segmentation_loss)
    epoch_summary = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "val_loss": val_summary["loss"],
        "val_iou": val_summary["iou"],
        "val_dice": val_summary["dice"],
        "val_boundary_f1": val_summary["boundary_f1"],
    }
    history.append(epoch_summary)
    print(epoch_summary)

    if val_summary["iou"] > best_val_iou:
        best_val_iou = val_summary["iou"]
        torch.save(model.state_dict(), best_checkpoint_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_axis = [item["epoch"] for item in history]
axes[0].plot(epochs_axis, [item["train_loss"] for item in history], marker="o", label="train")
axes[0].plot(epochs_axis, [item["val_loss"] for item in history], marker="o", label="val")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs_axis, [item["val_iou"] for item in history], marker="o", label="Val IoU")
axes[1].plot(epochs_axis, [item["val_dice"] for item in history], marker="o", label="Val Dice")
axes[1].plot(epochs_axis, [item["val_boundary_f1"] for item in history], marker="o", label="Val Boundary F1")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
fig.tight_layout()
training_curve_path = ARTIFACTS_DIR / "training_curves.png"
fig.savefig(training_curve_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print("Best checkpoint:", best_checkpoint_path)
print("Saved:", training_curve_path.name)

## Training Loop or Fine-Tuning Pipeline

The model is trained with a combined BCE-with-logits plus Dice loss. Validation tracks both region overlap and boundary quality so the notebook does not overstate success from coarse masks.

## Inference Examples

After training, we reload the best checkpoint and visualize predictions on the held-out test split.

## Evaluation

Primary metrics:

- IoU for region overlap
- Dice for foreground agreement
- Boundary F1 for edge quality

Boundary F1 is especially relevant for footprint extraction because a mask can score reasonably on area overlap while still missing roof edges or expanding into nearby roads and trees.

In [ ]:
model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))

test_detailed_records = []
with torch.no_grad():
    model.eval()
    for images, masks, names in test_loader:
        images_device = images.to(DEVICE)
        logits = model(images_device)
        preds = (torch.sigmoid(logits) > 0.5).float().cpu()
        for index, name in enumerate(names):
            image_np = denormalize_image(images[index])
            pred_mask = preds[index, 0].numpy().astype(np.uint8)
            true_mask = masks[index, 0].numpy().astype(np.uint8)
            iou, dice = binary_iou_dice(pred_mask, true_mask)
            boundary_f1 = boundary_f1_score(pred_mask, true_mask)
            test_detailed_records.append(
                {
                    "name": name,
                    "iou": iou,
                    "dice": dice,
                    "boundary_f1": boundary_f1,
                    "image": image_np,
                    "pred_mask": pred_mask,
                    "true_mask": true_mask,
                }
            )

test_summary = {
    "iou": float(np.mean([record["iou"] for record in test_detailed_records])),
    "dice": float(np.mean([record["dice"] for record in test_detailed_records])),
    "boundary_f1": float(np.mean([record["boundary_f1"] for record in test_detailed_records])),
}

baseline_test_records = []
for image_path, mask_path in split_pairs["test"]:
    image = load_rgb_image(image_path)
    true_mask = load_binary_mask(mask_path)
    image_resized = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
    mask_resized = cv2.resize(true_mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
    pred_mask = grayscale_baseline_mask(image_resized, reference_coverage)
    iou, dice = binary_iou_dice(pred_mask, mask_resized)
    baseline_test_records.append(
        {
            "name": image_path.name,
            "iou": iou,
            "dice": dice,
            "boundary_f1": boundary_f1_score(pred_mask, mask_resized),
        }
    )

baseline_test_summary = {
    "iou": float(np.mean([record["iou"] for record in baseline_test_records])),
    "dice": float(np.mean([record["dice"] for record in baseline_test_records])),
    "boundary_f1": float(np.mean([record["boundary_f1"] for record in baseline_test_records])),
}

example_records = test_detailed_records[:4]
fig, axes = plt.subplots(len(example_records), 4, figsize=(14, 3 * len(example_records)))
for row, record in enumerate(example_records):
    axes[row, 0].imshow(record["image"])
    axes[row, 0].set_title(f"RGB\n{record['name']}")
    axes[row, 1].imshow(record["true_mask"], cmap="gray")
    axes[row, 1].set_title("Ground Truth")
    axes[row, 2].imshow(record["pred_mask"], cmap="gray")
    axes[row, 2].set_title(f"Prediction\nIoU={record['iou']:.3f}")
    axes[row, 3].imshow(overlay_mask(record["image"], record["pred_mask"]))
    axes[row, 3].set_title(f"Overlay\nBoundary F1={record['boundary_f1']:.3f}")
    for col in range(4):
        axes[row, col].axis("off")
fig.tight_layout()
qualitative_path = ARTIFACTS_DIR / "qualitative_masks.png"
fig.savefig(qualitative_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print("Baseline test summary:")
print(json.dumps(baseline_test_summary, indent=2))
print("Model test summary:")
print(json.dumps(test_summary, indent=2))
print("Saved:", qualitative_path.name)

## Error Analysis

We sort the held-out predictions by boundary F1 and inspect the worst cases. This focuses the analysis on boundary failures rather than only on area mismatch.

## Boundary Failure Analysis

A good footprint model should keep sharp edges, corners, and roof extents. We visualize two boundary error types:

- missed edges: true boundaries with no nearby predicted boundary
- spurious edges: predicted boundaries not supported by the target mask

In [ ]:
worst_boundary_records = sorted(test_detailed_records, key=lambda record: record["boundary_f1"])[:4]

fig, axes = plt.subplots(len(worst_boundary_records), 5, figsize=(18, 4 * len(worst_boundary_records)))
for row, record in enumerate(worst_boundary_records):
    image = record["image"]
    pred_mask = record["pred_mask"]
    true_mask = record["true_mask"]
    pred_boundary = boundary_mask(pred_mask)
    true_boundary = boundary_mask(true_mask)
    kernel = np.ones((2 * BOUNDARY_TOLERANCE + 1, 2 * BOUNDARY_TOLERANCE + 1), np.uint8)
    missed = (true_boundary == 1) & (cv2.dilate(pred_boundary, kernel, iterations=1) == 0)
    spurious = (pred_boundary == 1) & (cv2.dilate(true_boundary, kernel, iterations=1) == 0)
    boundary_vis = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
    boundary_vis[missed] = np.array([255, 0, 0], dtype=np.uint8)
    boundary_vis[spurious] = np.array([0, 255, 255], dtype=np.uint8)

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"RGB\n{record['name']}")
    axes[row, 1].imshow(true_mask, cmap="gray")
    axes[row, 1].set_title("Ground Truth")
    axes[row, 2].imshow(pred_mask, cmap="gray")
    axes[row, 2].set_title("Prediction")
    axes[row, 3].imshow(boundary_vis)
    axes[row, 3].set_title("Boundary Errors")
    axes[row, 4].imshow(overlay_mask(image, true_mask))
    axes[row, 4].imshow(np.ma.masked_where(boundary_vis.sum(axis=2) == 0, boundary_vis), alpha=0.7)
    axes[row, 4].set_title(
        f"Overlay\nIoU={record['iou']:.3f}, Dice={record['dice']:.3f}, BF1={record['boundary_f1']:.3f}"
    )
    for col in range(5):
        axes[row, col].axis("off")
fig.tight_layout()
boundary_path = ARTIFACTS_DIR / "boundary_failure_analysis.png"
fig.savefig(boundary_path, dpi=150, bbox_inches="tight")
plt.close(fig)

names = [record["name"] for record in test_detailed_records]
ious = [record["iou"] for record in test_detailed_records]
dices = [record["dice"] for record in test_detailed_records]
boundary_scores = [record["boundary_f1"] for record in test_detailed_records]

fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(len(names))
ax.plot(positions, ious, marker="o", label="IoU")
ax.plot(positions, dices, marker="o", label="Dice")
ax.plot(positions, boundary_scores, marker="o", label="Boundary F1")
ax.set_xticks(positions)
ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Per-Image Test Metrics")
ax.legend()
fig.tight_layout()
per_image_path = ARTIFACTS_DIR / "per_image_metrics.png"
fig.savefig(per_image_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print("Worst boundary cases:", [record["name"] for record in worst_boundary_records])
print("Saved:", boundary_path.name)
print("Saved:", per_image_path.name)

## Limitations

- The validation split is very small, so a few tiles can move the validation metrics substantially.
- Resizing from 1500 x 1500 to 256 x 256 speeds up training but softens fine roof boundaries.
- A lightweight encoder is a runtime compromise, not the strongest possible model.

## How to Improve This Project

1. Train on random high-resolution crops instead of full-image resize only.
2. Increase epochs and add early stopping on a larger validation split.
3. Try `DeepLabV3Plus` or a stronger SMP encoder when GPU memory allows.
4. Add post-processing or polygon regularization before exporting GIS-ready footprints.

In [ ]:
metrics = {
    "dataset": "Massachusetts Buildings",
    "dataset_id": DATASET_ID,
    "dataset_root": str(dataset_root),
    "split_counts": split_counts,
    "image_size": IMAGE_SIZE,
    "epochs": EPOCHS,
    "device": DEVICE,
    "model": f"smp.Unet({ENCODER_NAME})",
    "baseline_val": baseline_summary,
    "baseline_test": baseline_test_summary,
    "best_val_iou": float(best_val_iou),
    "test": test_summary,
    "history": history,
    "worst_boundary_cases": [
        {
            "name": record["name"],
            "iou": record["iou"],
            "dice": record["dice"],
            "boundary_f1": record["boundary_f1"],
        }
        for record in worst_boundary_records
    ],
}
metrics_path = BASE_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2))
print("Saved:", metrics_path.name)

## Production Considerations

- Export masks at the original resolution if the downstream system expects accurate parcel boundaries.
- Track both region overlap and boundary metrics in monitoring because coarse masks can hide in average IoU.
- Add tiling with overlap for large aerial scenes to reduce edge artifacts.
- Save vectorized polygons only after geometry cleanup and topology checks.

## Common Mistakes

- Resizing masks with bilinear interpolation instead of nearest neighbor.
- Reporting only IoU and ignoring edge quality.
- Comparing a model against a baseline at a different image resolution.
- Treating white PNG masks as multiclass labels when they are binary foreground masks.

In [ ]:
result_delta = {
    "iou_gain_over_baseline": float(test_summary["iou"] - baseline_test_summary["iou"]),
    "dice_gain_over_baseline": float(test_summary["dice"] - baseline_test_summary["dice"]),
    "boundary_f1_gain_over_baseline": float(test_summary["boundary_f1"] - baseline_test_summary["boundary_f1"]),
}
print("Improvement over baseline:")
print(json.dumps(result_delta, indent=2))
print("Artifacts:")
for path in sorted(ARTIFACTS_DIR.iterdir()):
    print("-", path.name)
print("Checkpoint:", best_checkpoint_path.name)

## Mini Challenge / Exercises

1. Replace U-Net with `smp.DeepLabV3Plus` and compare the change in boundary F1.
2. Train on random 512 x 512 crops and evaluate whether the sharper local context improves footprint edges.
3. Export predicted masks to polygons and inspect which geometric cleanup rules are still needed.

## Final Summary / Key Takeaways

This notebook rebuilt the project around the requested stack: PyTorch, segmentation_models_pytorch, and Albumentations on the real Massachusetts Buildings dataset. The evaluation reports IoU and Dice for overlap quality, and boundary F1 plus worst-case boundary visualizations for edge failure analysis. That combination is more honest for footprint extraction than overlap metrics alone.